# Project 59 — Local CRM Enrichment Agent
## Account Analysis → Risk Scoring → Action Plans → Email Drafts

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — CRM Data

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

accounts = [
    {"name": "Acme Corp", "industry": "Manufacturing", "arr": 50000,
     "renewal": "2025-06-01", "health": "at-risk", "last_contact": "45 days ago",
     "open_tickets": 3, "nps": 25,
     "notes": "Unhappy with recent downtime. Considering competitor. Champion left the company."},
    {"name": "TechStart Inc", "industry": "SaaS", "arr": 12000,
     "renewal": "2025-09-01", "health": "healthy", "last_contact": "7 days ago",
     "open_tickets": 0, "nps": 85,
     "notes": "Interested in upgrading to enterprise. Champion: CTO Sarah. Expanding team."},
    {"name": "GlobalRetail", "industry": "Retail", "arr": 95000,
     "renewal": "2025-04-15", "health": "at-risk", "last_contact": "30 days ago",
     "open_tickets": 5, "nps": 40,
     "notes": "Integration issues with their POS system. Budget review happening next month."},
    {"name": "DataDriven Co", "industry": "Analytics", "arr": 28000,
     "renewal": "2025-12-01", "health": "healthy", "last_contact": "14 days ago",
     "open_tickets": 1, "nps": 72,
     "notes": "Using 60% of features. Potential for add-on module."},
]
print(f"CRM: {len(accounts)} accounts, total ARR: ${sum(a['arr'] for a in accounts):,}")

## Step 2 — Risk Scoring Engine

In [ ]:
class RiskAssessment(BaseModel):
    account: str
    risk_score: float = Field(ge=0, le=1, description="0=safe, 1=critical risk")
    risk_factors: list[str]
    churn_probability: str = Field(description="low, medium, high, critical")
    revenue_at_risk: float
    days_to_renewal: int

risk_scorer = llm.with_structured_output(RiskAssessment)

assessments = []
for acct in accounts:
    assessment = risk_scorer.invoke(
        f"Assess churn risk for this account (today is 2025-02-01):\n{json.dumps(acct, indent=2)}"
    )
    assessments.append(assessment)
    icon = {"low":"🟢","medium":"🟡","high":"🔴","critical":"⛔"}[assessment.churn_probability]
    print(f"  {icon} {assessment.account:<20} risk={assessment.risk_score:.0%} "
          f"churn={assessment.churn_probability} ARR@risk=${assessment.revenue_at_risk:,.0f}")

## Step 3 — Action Plan Generator

In [ ]:
class ActionPlan(BaseModel):
    account: str
    immediate_actions: list[str]
    thirty_day_plan: list[str]
    talking_points: list[str]
    upsell_opportunity: str
    recommended_contact: str = Field(description="Who to contact and when")

planner_llm = llm.with_structured_output(ActionPlan)

for acct, assessment in zip(accounts, assessments):
    plan = planner_llm.invoke(
        f"Create an action plan for this account:\n"
        f"Account: {json.dumps(acct)}\nRisk: {assessment.model_dump()}"
    )
    print(f"\n{'='*50}")
    print(f"ACTION PLAN — {plan.account}")
    print(f"Contact: {plan.recommended_contact}")
    print(f"\nImmediate:")
    for a in plan.immediate_actions:
        print(f"  → {a}")
    print(f"\n30-Day Plan:")
    for a in plan.thirty_day_plan:
        print(f"  📅 {a}")
    print(f"\nUpsell: {plan.upsell_opportunity}")

## Step 4 — Generate Outreach Emails

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a professional outreach email from a Customer Success Manager. "
     "Be warm, reference specific issues, and propose a call."),
    ("human", "Account: {account}\nHealth: {health}\nKey issues: {notes}\n"
     "NPS: {nps}\nAction: {action}")
])
email_chain = email_prompt | llm | StrOutputParser()

# Generate emails for at-risk accounts
for acct in accounts:
    if acct["health"] != "at-risk":
        continue
    email = email_chain.invoke({
        "account": acct["name"],
        "health": acct["health"],
        "notes": acct["notes"],
        "nps": acct["nps"],
        "action": "Schedule a review call to address concerns",
    })
    print(f"\n{'='*50}")
    print(f"EMAIL TO: {acct['name']}")
    print("=" * 50)
    print(email[:500])

## What You Learned
- **CRM risk scoring** with structured analysis
- **Churn prediction** from account signals
- **Action plan generation** with timelines
- **Automated outreach email drafting**